***

Preparing Workspace

***

In [ ]:
# Packages
import pandas as pd
import numpy as np
import os
from pathlib import Path
import re
from datetime import datetime
pd.set_option('display.max_columns', None)

In [ ]:
# Calm luh test run
# Need to use the criteria dataset, and the original. 
# No exports on this one. 
# Data

path_census = "I:\\Projects\\Warren\\Environmental_Justice_March_2023\\SACOG_EJ_UPDATE_2024\\Python\\YOUR_OUTPUT_FOLDER"
workbook_census = "Block Groups Processed - Ready for Calculator.xlsx"
sheet_census = 'ready_to_calculate'



df_census = pd.read_excel(os.path.join(path_census, workbook_census), sheet_name=sheet_census)


input_directory = Path('I:/Projects/Warren/Environmental_Justice_March_2023/SACOG_EJ_UPDATE_2024/EJ_2022/CES4')
year = '2022'
input_file = input_directory / f'SACOG_{year}_Criteria_wCES.csv' # Adjust based on actual input file naming
df_ces4 = pd.read_csv(input_file)


display(df_census.head(3)); print('')
display(df_ces4.head(3))

In [ ]:
df_census = df_census.merge(df_ces4[['Geography', 'Geographic Area Name', 'CES4_designated']], how = 'left',
                left_on = 'NAME', right_on = 'Geographic Area Name').drop(columns = ['NAME'])

df_census = df_census.dropna()
df_census = df_census.drop(['Geographic Area Name', 'State FIPS', 'County FIPS'], axis = 1)

# Specify the columns you want to move to the left
columns_to_move = ['Geography', 'CES4_designated']

# Create a new column order
new_column_order = columns_to_move + [col for col in df_census.columns if col not in columns_to_move]

# Reorder the DataFrame
df_census = df_census[new_column_order]

df_census.head()

In [ ]:
df_census['HousingBurden'] = (df_census['GrossMortgage_50 pct or more'] + df_census['GrossRent_50 pct or more'])
df_census['HousingBurden_Total'] = (df_census['GrossMortgage_Total'] + df_census['GrossRent_Total'])
df_census = df_census.drop(['GrossMortgage_Total', 'GrossRent_Total', 'GrossMortgage_50 pct or more', 'GrossRent_50 pct or more'], axis=1)

df_census.head()

***

Calculator

***

In [ ]:
def assign_ej(df, quantile_val):
    # This might have to change depending on the structure of the input dataframe.
    prefixes = set([col.split('_')[0] for col in list(df.columns[10:])])
    print(prefixes)

    for prefix in prefixes:    
        marked_cols = df.filter(like=prefix).columns.tolist()

        total_col = [col for col in marked_cols if col.endswith('Total')]
        non_total_col = [col for col in marked_cols if not col.endswith('Total')]

        if total_col and non_total_col:
            total_col = total_col[0]
            non_total_col = non_total_col[0]
        else:
            raise ValueError(f"Could not find total and non-total columns for prefix: {prefix}.")
        
        percent_col = f"{prefix}_Percent"

        df[percent_col] = (df[non_total_col].fillna(0) / df[total_col].fillna(1)) * 100
        threshold = df[percent_col].dropna().quantile(quantile_val)
        print(f"Threshold for", prefix, "is: ", threshold)
        df[f"{prefix}_Marked"] = 0
        df.loc[df[percent_col] >= threshold, f"{prefix}_Marked"] = 1

    df['Minority_Percent'] = 100 - df['Minority_Percent']
    threshold = df['Minority_Percent'].dropna().quantile(0.75) # in Second calculator script we used 0.7
    df["Minority_Marked"] = 0
    df.loc[df["Minority_Percent"] >= threshold, "Minority_Marked"] = 1

    df['LowIncome_Percent'] = 100 - df['LowIncome_Percent']
    threshold = df['LowIncome_Percent'].dropna().quantile(0.75) # in second calculator script we used 0.45
    df["LowIncome_Marked"] = 0
    df.loc[df["LowIncome_Percent"] >= threshold, "LowIncome_Marked"] = 1

    ################### Labelling #######################################

    category_columns = [col for col in df.columns if '_Marked' in col]

    for col in category_columns:
        col_base = col.replace('_Marked', '_identified')
        df[col_base] = (df[col] == 1).astype(int)
    
    other_factors_columns = [col for col in category_columns if col not in ['Minority_Marked', 'Low_Income_Marked']]
    df['Other_factors_Summed'] = df[other_factors_columns].sum(axis=1)

    print('')
    print(df['Other_factors_Summed'].value_counts())

    df['Other_factors_identified'] = (df['Other_factors_Summed'] >= 4).astype(int)

    identified_columns = [col for col in df.columns if '_identified' in col]
    df['All_categories_summed'] = df[category_columns].sum(axis=1)
    df['All_identified_summed'] = df[identified_columns].sum(axis=1)

    ############################### Final Criteria #####################################

    # Define the categories as before
    categories = [
        ('Highest Priority'     , ['Minority_identified' , 'LowIncome_identified', 'Other_factors_identified', 'CES4_designated'] ),
        ('LowInc/Min/Oth'       , ['Minority_identified' , 'LowIncome_identified', 'Other_factors_identified'                   ] ),
        ('LowInc/Min/CES'       , ['Minority_identified' , 'LowIncome_identified',                             'CES4_designated'] ),
        ('LowInc/Min'           , ['Minority_identified' , 'LowIncome_identified'                                               ] ),
        ('LowInc/Oth'           , [                        'LowIncome_identified', 'Other_factors_identified'                   ] ),
        ('Min/Oth'              , ['Minority_identified' ,                         'Other_factors_identified'                   ] ),
        ('LowInc or Min and CES', ['Minority_identified' , 'LowIncome_identified',                             'CES4_designated'] ),  # Adjusted case
        ('Minority'             , ['Minority_identified'                                                                        ] ),
        ('Low Income'           , [                        'LowIncome_identified'                                               ] ),
        ('Other factors'        , [                                                'Other_factors_identified'                   ] ),
        ('CES4'                 , [                                                                            'CES4_designated'] )
    ]

    # Making the categories, and then converting to wide. 
    for category, factors in categories:
        df[category] = 0

        if category == 'LowInc or Min and CES':
            # In the code, there was a special case for this one where it had to be an or, this is just to catch this correctly
            condition = ((df['LowIncome_identified'] == 1) | (df['Minority_identified'] == 1)) & (df['CES4_designated'] == 1)
        else:
            # General case: all factors must be 1 in order to be labelled
            condition = np.all([df[factor] == 1 for factor in factors], axis=0)
        
        # Assign 1 where the condition is True
        df.loc[condition, category] = 1

        # Define conditions and choices for labeling
    conditions = [
        ( df['Minority_identified'  ] == 1) & (df['LowIncome_identified'    ] == 1)  & (df['Other_factors_identified'] == 1) & (df['CES4_designated'] == 1), # Highest Priority
        ( df['LowIncome_identified' ] == 1) & (df['Minority_identified'     ] == 1)  & (df['Other_factors_identified'] == 1)                               , # LowInc/Min/Oth
        ( df['LowIncome_identified' ] == 1) & (df['Minority_identified'     ] == 1)  & (df['CES4_designated'         ] == 1)                               , # LowInc/Min/CES
        ( df['LowIncome_identified' ] == 1) & (df['Minority_identified'     ] == 1)                                                                        , # LowInc/Min
        ( df['LowIncome_identified' ] == 1) & (df['Other_factors_identified'] == 1)                                                                        , # LowInc/Oth
        ( df['Minority_identified'  ] == 1) & (df['Other_factors_identified'] == 1)                                                                        , # Min/Oth
        ((df['LowIncome_identified' ] == 1) | (df['Minority_identified'     ] == 1)) & (df['CES4_designated'] == 1)                                        , # LowInc or Min and CES
        ( df['Minority_identified'  ] == 1) & (df['LowIncome_identified'    ] != 1)  & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] != 1), # Minority
        ( df['Minority_identified'  ] != 1) & (df['LowIncome_identified'    ] == 1)  & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] != 1), # Low Income
        ( df['Minority_identified'  ] != 1) & (df['LowIncome_identified'    ] != 1)  & (df['Other_factors_identified'] == 1) & (df['CES4_designated'] != 1), # Other factors
        ( df['Minority_identified'  ] != 1) & (df['LowIncome_identified'    ] != 1)  & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] == 1)  # CES4
    ]
    choices = ['Highest Priority', 'LowInc/Min/Oth', 'LowInc/Min/CES', 'LowInc/Min', 'LowInc/Oth', 'Min/Oth',
                'LowInc or Min and CES', 'Minority', 'Low Income', 'Other factors', 'CES4']

    # Apply conditions to assign labels
    df['EJ_Label'] = np.select(conditions, choices, default='')

    # Now we subset the dataframe to only keep the first six columns and the condition columns
    # meta_list = ['Geography', 'Geographic Area Name', 'State FIPS', 'County FIPS', 
    #              'County Name', 'Tract ID', 'Block Group ID', 'Year']
    # meta_cols = [col for col in meta_list if col in df.columns]
    # cat_cols = [category[0] for category in categories]
    # wish_cols = meta_cols + cat_cols
    
    # # If the wish cols are found in the df, we will keep em. If not, we proceed without.
    # keep_cols = list(dict.fromkeys([col for col in wish_cols if col in df.columns]))

    # # Subsetting dataframe now
    # df = df[keep_cols]

    return df

df_ej = assign_ej(df_census, quantile_val= 0.75)
print('')
print(df_ej['EJ_Label'].value_counts())
print('')
print(df_ej['Highest Priority'].value_counts())
print('')
df_ej.head()

In [ ]:
# df_ej.to_excel(os.path.join(path_census, 'Block Groups Processed - EJ Assigned.xlsx'))

***

QC

***

In [ ]:
path_down = r'C:\Users\jfontes\Downloads'
df_warren = pd.read_csv(os.path.join(path_down, 'SACOG_EJ_2022_BG_TABLE.csv'))

high_priorities = df_ej[df_ej['Highest Priority'] == 1]
high_priorities = high_priorities.Geography.unique()

df_warren = df_warren[df_warren['SACOG_2022_Final_csv_Geography'].isin(high_priorities)]
df_warren = df_warren[df_warren['SACOG_2022_Final_csv_EJ_Label'] != 'Highest Priority']

discrepancies = df_warren['SACOG_2022_Final_csv_Geography'].unique()

df_warren = df_warren.sort_values('SACOG_2022_Final_csv_Geography')

display(df_warren.head())


In [ ]:
df_ej = df_ej.sort_values('Geography')
df_ej[df_ej['Geography'].isin(discrepancies)].head()
